## RQ3: Table 9 - Application-level source coverage by PII type

In [ ]:
import json
import os
import glob
import pandas as pd
from collections import defaultdict
import re

In [ ]:
GPT4O_RESULTS_DIR = '../normalized_PII_results/gpt4o/db_level/'
GROUND_TRUTH_DIR = '../normalized_PII_results/ground_truth/db_level/'

PII_TYPES = ['EMAIL', 'PHONE', 'USERNAME', 'PERSON_NAME', 'POSTAL_ADDRESS']

APP_MAPPING = {
    'A1': 'WhatsApp',
    'A2': 'Snapchat',
    'A3': 'Telegram',
    'A4': 'Google Maps',
    'A5': 'Samsung Internet',
    'I1': 'WhatsApp (iOS)',
    'I2': 'Contacts',
    'I3': 'Apple Messages',
    'I4': 'Safari',
    'I5': 'Calendar'
}

COLUMN_MAPPING = {
    'EMAIL': 'Email',
    'PHONE': 'Phone',
    'USERNAME': 'User Name',
    'PERSON_NAME': 'Person Name',
    'POSTAL_ADDRESS': 'Postal Address'
}

In [ ]:
def parse_filename(filepath):
    """Parses a filename to extract the app ID and database name."""
    base_name = os.path.basename(filepath)
    # Format: PII_{APP_ID}_{DB_NAME}_{TIMESTAMP}.jsonl
    match = re.match(r'PII_([A-Z0-9]+)_(.*)_\d{8}T\d{6}Z\.jsonl', base_name)
    if match:
        app_id = match.group(1)
        db_name = match.group(2)
        return app_id, db_name
    return None, None

def load_data(path):
    """Loads PII presence data from a directory of jsonl files."""
    # Structure: {app_id: {db_name: {pii_type: has_pii_bool}}}
    data = defaultdict(lambda: defaultdict(lambda: defaultdict(bool)))
    files = glob.glob(os.path.join(path, '*.jsonl'))
    for f_path in files:
        app_id, db_name = parse_filename(f_path)
        if not app_id or not db_name:
            continue
        with open(f_path, 'r') as f:
            for line in f:
                record = json.loads(line)
                pii_type = record['PII_type']
                if len(record['PII']) > 0:
                    data[app_id][db_name][pii_type] = True
    return data

gt_data = load_data(GROUND_TRUTH_DIR)
system_data = load_data(GPT4O_RESULTS_DIR)

In [ ]:
table_data = []

for app_id, app_name in APP_MAPPING.items():
    row = {'ID': app_id, 'Application': app_name}
    
    app_dbs_in_gt = gt_data.get(app_id, {}).keys()

    # --- Per-PII Type Calculation ---
    for pii_type in PII_TYPES:
        col_name = COLUMN_MAPPING[pii_type]
        
        # DG(a,t): set of databases for app 'a' that contain pii_type 't' in ground truth
        gt_dbs_with_pii = {db for db in app_dbs_in_gt if gt_data.get(app_id, {}).get(db, {}).get(pii_type, False)}
        
        # DS(a,t): set of databases for app 'a' that contain pii_type 't' in system output
        system_dbs_with_pii = {db for db in app_dbs_in_gt if system_data.get(app_id, {}).get(db, {}).get(pii_type, False)}
        
        gt_count = len(gt_dbs_with_pii)
        
        if gt_count == 0:
            row[col_name] = '-'
        else:
            # covered = |DG(a,t) ∩ DS(a,t)|
            covered_count = len(gt_dbs_with_pii.intersection(system_dbs_with_pii))
            row[col_name] = f"{covered_count}/{gt_count}"

    # --- All PII Calculation ---
    # Databases in GT for this app that have *any* PII type
    gt_dbs_with_any_pii = {
        db for db in app_dbs_in_gt 
        if any(gt_data.get(app_id, {}).get(db, {}).get(pt, False) for pt in PII_TYPES)
    }
    
    # Databases in system output for this app that have *any* PII type
    system_dbs_with_any_pii = {
        db for db in app_dbs_in_gt
        if any(system_data.get(app_id, {}).get(db, {}).get(pt, False) for pt in PII_TYPES)
    }

    all_gt_count = len(gt_dbs_with_any_pii)
    if all_gt_count == 0:
        row['All PII'] = '-'
    else:
        all_covered_count = len(gt_dbs_with_any_pii.intersection(system_dbs_with_any_pii))
        row['All PII'] = f"{all_covered_count}/{all_gt_count}"
        
    table_data.append(row)

In [ ]:
df = pd.DataFrame(table_data)

# Reorder columns to match Table 9
final_columns = ['ID', 'Application'] + [COLUMN_MAPPING[pt] for pt in PII_TYPES] + ['All PII']
df = df[final_columns]

df = df.set_index('ID')

# Display the dataframe
df

In [ ]:
# Optional: Save to LaTeX
latex_output = df.to_latex(index=True, caption='Application-level source coverage by PII type.', label='tab:app_level_coverage', column_format='ll' + 'c' * (len(df.columns)))
print(latex_output)